In [45]:
!pip install emoji

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# move to the current directory
os.chdir('/content/drive/My Drive/Thesis_Repository/Final_Google_Drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Datasets
- Eedi
    - Personal and Sensitive Information
    - https://huggingface.co/datasets/Eedi/Question-Anchored-Tutoring-Dialogues-2k
    - Paper : https://aclanthology.org/2025.emnlp-main.1397.pdf
    - This dataset contains real student-tutor dialogues. Extensive efforts were made to mitigate privacy risks, but these risks cannot be alleviated completely. Identifying details were removed or anonymized where detected, but some sensitive content may persist. If you have concerns about any content, please contact the first author at matthew.zent@eedi.co.uk.

- MathtutorMR
    - https://www.mikeion.com/
    - MATHMENTORDB https://huggingface.co/datasets/mikeion/mathconverse_pseudonyms
    - https://dl.acm.org/doi/epdf/10.1145/3774398.3811608
    - https://openreview.net/pdf?id=hw76uAXOgg

## 1. load data


In [ ]:
from huggingface_hub import notebook_login
from huggingface_hub import login
# notebook_login()

login(token="[HF Token]")

In [48]:
import pandas as pd

mathtutormr_df = pd.read_parquet("hf://datasets/mikeion/mathconverse_pseudonyms/all-help-channels-expanded.parquet")

In [ ]:
# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'anchored-dialogues/train-00000-of-00001.parquet', 'test': 'anchored-dialogues/test-00000-of-00001.parquet'}
Eedi_df = pd.read_parquet("hf://datasets/Eedi/Question-Anchored-Tutoring-Dialogues-2k/" + splits["train"]) + splits["test"]

In [50]:
# Select the necessary columns and sort by conversation_id and timestamp
mathtutormr_df_ = mathtutormr_df[["conversation_id", "helper", "content", "timestamp"]]
mathtutormr_df_ = mathtutormr_df_.sort_values(["conversation_id", "timestamp"])

# Add a new column MessageSequence that is the cumulative count of turns in each conversation
mathtutormr_df_["message_sequence"] = mathtutormr_df_.groupby("conversation_id").cumcount() + 1
mathtutormr_df_ = mathtutormr_df_[["conversation_id", "helper", "content", "message_sequence"]]
mathtutormr_df_.rename(columns={"helper": "is_tutor"}, inplace=True)
mathtutormr_df_


,conversation_id,is_tutor,content,message_sequence
0,1,0,,1
1,1,1,simplification?,2
2,1,0,yes I'm too confused to understand this,3
3,1,1,how about combining all of those constants first?,4
4,1,1,"looks like you have a $12$, a $- \frac{2}{5}$ ...",5
...,...,...,...,...
5450292,205884,0,.close,10
5450293,205885,0,https://cdn.discordapp.com/attachments/9267015...,1
5450294,205885,0,hey what does this notation mean,2
5450295,205885,0,is it another way to write a vector,3


In [51]:
Eedi_df_ = Eedi_df[["InterventionId", "IsTutor", "MessageString", "MessageSequence"]]
Eedi_df_.rename(columns={"MessageString": "content", "InterventionId": "conversation_id", "MessageSequence": "message_sequence", "IsTutor": "is_tutor"}, inplace=True)
Eedi_df_

/tmp/ipykernel_6459/3528149072.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Eedi_df_.rename(columns={"MessageString": "content", "InterventionId": "conversation_id", "MessageSequence": "message_sequence", "IsTutor": "is_tutor"}, inplace=True)


,conversation_id,is_tutor,content,message_sequence
0,10,1,"Hello Lina, just wanted to check, you OK?",1
1,10,1,You don't have to have help if you don't want ...,2
2,10,0,Hi I would you preferred to be called Lina Chen,3
3,10,0,I need help,4
4,10,0,On this question I am a bit stuck at this can ...,5
...,...,...,...,...
55317,12486,0,Ok,20
55318,12486,1,Happy with this now/,21
55319,12486,1,/,22
55320,12486,1,"Ah, I meant a question mark sorry!",23


## 2. Normalize Content

In [52]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import clean_EEDI_MathtutorMR, delete_MathtutorMR_rows_with_close

#### MathtutorMR

In [53]:
clean_mathtutormr_df = clean_EEDI_MathtutorMR(mathtutormr_df_)
clean_mathtutormr_df = delete_MathtutorMR_rows_with_close(clean_mathtutormr_df)

In [54]:
clean_mathtutormr_df

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,1,1,simplification?,2,1
1,1,0,yes i'm too confused to understand this,3,2
2,1,1,how about combining all of those constants first?,4,3
3,1,1,"looks like you have a 12 , a - 2 5 and a 3",5,4
4,1,1,what do you get when you multiply all those to...,6,5
...,...,...,...,...,...
5047232,205884,0,got it,7,7
5047233,205884,0,thank you!,8,8
5047234,205885,0,hey what does this notation mean,2,1
5047235,205885,0,is it another way to write a vector,3,2


In [55]:
# check the first conversation with the first conversation_id
first_conversation_id = clean_mathtutormr_df["conversation_id"].drop_duplicates().iloc[0]
clean_mathtutormr_df[clean_mathtutormr_df["conversation_id"] == first_conversation_id]

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,1,1,simplification?,2,1
1,1,0,yes i'm too confused to understand this,3,2
2,1,1,how about combining all of those constants first?,4,3
3,1,1,"looks like you have a 12 , a - 2 5 and a 3",5,4
4,1,1,what do you get when you multiply all those to...,6,5
5,1,0,36?,7,6
6,1,1,that doesnt sound right thonk,8,7
7,1,1,i think 12 3 is gonna be 36,9,8
8,1,1,but then you also need to multiply in the -2 5,10,9
9,1,1,i'm suggesting to move the constants just beca...,12,10


In [56]:
# check the second conversation with the second conversation_id
second_conversation_id = clean_mathtutormr_df["conversation_id"].drop_duplicates().iloc[1]
clean_mathtutormr_df[clean_mathtutormr_df["conversation_id"] == second_conversation_id]

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
15,2,0,how'd i find the stationary point for this fun...,1,1
16,2,0,i'll take the derivative and set it to zero,2,2
17,2,0,but i don't remember how to get the roots of a...,3,3
18,2,1,that cubic would not be that hard to factor.,4,4
19,2,0,"my first idea was to factor x 2 out, i.e. x 2 ...",5,5
20,2,0,but it feels like a useless maneuver,6,6
21,2,1,that -9 will disappear,7,7
22,2,1,you didn't differentiate properly,8,8
23,2,1,now the roots are clear,9,9
24,2,0,yeah,10,10


In [57]:
# check the second conversation with the second conversation_id
second_conversation_id = clean_mathtutormr_df["conversation_id"].drop_duplicates().iloc[1]
clean_mathtutormr_df[clean_mathtutormr_df["conversation_id"] == second_conversation_id]

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
15,2,0,how'd i find the stationary point for this fun...,1,1
16,2,0,i'll take the derivative and set it to zero,2,2
17,2,0,but i don't remember how to get the roots of a...,3,3
18,2,1,that cubic would not be that hard to factor.,4,4
19,2,0,"my first idea was to factor x 2 out, i.e. x 2 ...",5,5
20,2,0,but it feels like a useless maneuver,6,6
21,2,1,that -9 will disappear,7,7
22,2,1,you didn't differentiate properly,8,8
23,2,1,now the roots are clear,9,9
24,2,0,yeah,10,10


#### EEDI

In [58]:
clean_eedi_df = clean_EEDI_MathtutorMR(Eedi_df_)

/content/drive/MyDrive/Thesis_Repository/Final_Google_Drive/final__clean_transform_datasets.py:303: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["content"] = df["content"].map(normalize_text)


In [59]:
clean_eedi_df
clean_eedi_df[clean_eedi_df["conversation_id"] == clean_eedi_df["conversation_id"].iloc[0]]

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,10,1,"hello lina, just wanted to check, you ok?",1,1
1,10,1,you don't have to have help if you don't want ...,2,2
2,10,0,hi i would you preferred to be called lina chen,3,3
3,10,0,i need help,4,4
4,10,0,on this question i am a bit stuck at this can ...,5,5
5,10,0,thx u,6,6
6,10,1,"of course, thank you for correcting me on your...",7,7
7,10,0,ok thank you take your time,8,8
8,10,1,do you have any thoughts about what either ale...,9,9
9,10,1,do you think either of them is right or wrong?,10,10


## 3. Compress Datasets ( collapse messages with the same speakers )

In [60]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import compress_EEDI_MathtutorMR

In [61]:
compressed_mathtutormr_df = compress_EEDI_MathtutorMR(clean_mathtutormr_df)
compressed_eedi_df = compress_EEDI_MathtutorMR(clean_eedi_df)

##### MathtorMR

In [62]:
compressed_mathtutormr_df

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,1,1,simplification?,2,1
1,1,0,yes i'm too confused to understand this,3,2
2,1,1,how about combining all of those constants fir...,4,3
3,1,0,36?,7,4
4,1,1,that doesnt sound right thonk i think 12 3 is ...,8,5
...,...,...,...,...,...
2280732,205884,1,"you can't only do it there, but i'd suggest it",4,4
2280733,205884,0,i mean i can choose to do that,5,5
2280734,205884,1,yes,6,6
2280735,205884,0,got it thank you!,7,7


In [63]:
# check the second conversation with the first conversation_id
first_conversation_id = compressed_mathtutormr_df["conversation_id"].iloc[0]
compressed_mathtutormr_df[compressed_mathtutormr_df["conversation_id"] == first_conversation_id]

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,1,1,simplification?,2,1
1,1,0,yes i'm too confused to understand this,3,2
2,1,1,how about combining all of those constants fir...,4,3
3,1,0,36?,7,4
4,1,1,that doesnt sound right thonk i think 12 3 is ...,8,5
5,1,0,ok,17,6


In [64]:
# check the second conversation with the second conversation_id
first_conversation_id = compressed_mathtutormr_df["conversation_id"].iloc[1]
compressed_mathtutormr_df[compressed_mathtutormr_df["conversation_id"] == first_conversation_id]

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,1,1,simplification?,2,1
1,1,0,yes i'm too confused to understand this,3,2
2,1,1,how about combining all of those constants fir...,4,3
3,1,0,36?,7,4
4,1,1,that doesnt sound right thonk i think 12 3 is ...,8,5
5,1,0,ok,17,6


##### EEDI


In [65]:
compressed_eedi_df

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,10,1,"hello lina, just wanted to check, you ok? you ...",1,1
1,10,0,hi i would you preferred to be called lina che...,3,2
2,10,1,"of course, thank you for correcting me on your...",7,3
3,10,0,ok thank you take your time,8,4
4,10,1,do you have any thoughts about what either ale...,9,5
...,...,...,...,...,...
36541,12486,1,perfect,17,11
36542,12486,0,thankyou!,18,12
36543,12486,1,we'd have got the same answer if we just did 1...,19,13
36544,12486,0,ok,20,14


## 5. Filter conversation groups : When experimenting for all turns & preceding n turns
- the last turn should be a tutor
- the speakers should alternate in the same conversation id group
- the number of turns should be at least 3

In [66]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import filter_compressed_turns_EEDI_MathmentorMR, get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR, get_tutor_turn_only_EEDI_MathmentorMR

##### mathtutormr

In [67]:
filtered_compressed_mathtutormr_df = filter_compressed_turns_EEDI_MathmentorMR(compressed_mathtutormr_df)
filtered_compressed_mathtutormr_df

,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,1,1,simplification?,2,1
1,1,0,yes i'm too confused to understand this,3,2
2,1,1,how about combining all of those constants fir...,4,3
3,1,0,36?,7,4
4,1,1,that doesnt sound right thonk i think 12 3 is ...,8,5
...,...,...,...,...,...
2280730,205884,1,"not required, no",2,2
2280731,205884,0,so i can only do it to the secx 2 in the paren...,3,3
2280732,205884,1,"you can't only do it there, but i'd suggest it",4,4
2280733,205884,0,i mean i can choose to do that,5,5


##### EEDI

In [68]:
filtered_compressed_eedi_df = filter_compressed_turns_EEDI_MathmentorMR(compressed_eedi_df)
filtered_compressed_eedi_df


,conversation_id,is_tutor,content,message_sequence,new_message_sequence
0,10,1,"hello lina, just wanted to check, you ok? you ...",1,1
1,10,0,hi i would you preferred to be called lina che...,3,2
2,10,1,"of course, thank you for correcting me on your...",7,3
3,10,0,ok thank you take your time,8,4
4,10,1,do you have any thoughts about what either ale...,9,5
...,...,...,...,...,...
36541,12486,1,perfect,17,11
36542,12486,0,thankyou!,18,12
36543,12486,1,we'd have got the same answer if we just did 1...,19,13
36544,12486,0,ok,20,14


-----

## 6. Get all turns with or without speaker info
- boundry and speaker info can improve the further pretrained models => https://chatgpt.com/s/t_6a6c039427408191b51465f7782cd1d9

In [69]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR

#### mathtutormr

In [70]:
# Without Speaker
mathtutormr_df__all_turns__without_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_mathtutormr_df)
mathtutormr_df__all_turns__without_speaker_info

,text
0,simplification? </s>yes i'm too confused to un...
1,how'd i find the stationary point for this fun...
2,"hello, does anyone know the lagrange method? i..."
3,factor x 2 2x 1 how to do it </s>split the mid...
4,for question b. why is the first position 4? <...
...,...
115590,6 is in the wrong spot </s>ooh </s>so is 8 </s...
115591,i dont know where to start with this question?...
115592,im not sure what i need to do to solve this la...
115593,"hello, how is it possible to solve this linear..."


In [71]:
# With Speaker
mathtutormr_df__all_turns__with_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_mathtutormr_df, include_speaker=True)
mathtutormr_df__all_turns__with_speaker_info

,text
0,T: simplification? </s>S: yes i'm too confused...
1,S: how'd i find the stationary point for this ...
2,"S: hello, does anyone know the lagrange method..."
3,S: factor x 2 2x 1 how to do it </s>T: split t...
4,S: for question b. why is the first position 4...
...,...
115590,T: 6 is in the wrong spot </s>S: ooh </s>T: so...
115591,S: i dont know where to start with this questi...
115592,S: im not sure what i need to do to solve this...
115593,"S: hello, how is it possible to solve this lin..."


#### EEDI

In [72]:
# Without Speaker
eedi_df__all_turns__without_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_eedi_df)
eedi_df__all_turns__without_speaker_info

,text
0,"hello lina, just wanted to check, you ok? you ..."
1,"hi nathaniel, how are you today? </s>im good, ..."
2,"is it c </s>hi, another wordy one! one sec whi..."
3,"hi liam! </s>hi again </s>so, the range is... ..."
4,hi again zainab how can i help?! </s>i don't k...
...,...
1571,hi miriam </s>hello clara </s>how can i help?!...
1572,hi again sabrina how can i help?! </s>i'm a bi...
1573,hi morgan </s>hi </s>so for this one we need t...
1574,hi kofi i m lina one of the tutors at eedi can...


In [73]:
# With Speaker
eedi_df__all_turns__with_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_eedi_df, include_speaker=True)
eedi_df__all_turns__with_speaker_info

,text
0,"T: hello lina, just wanted to check, you ok? y..."
1,"T: hi nathaniel, how are you today? </s>S: im ..."
2,"S: is it c </s>T: hi, another wordy one! one s..."
3,"T: hi liam! </s>S: hi again </s>T: so, the ran..."
4,T: hi again zainab how can i help?! </s>S: i d...
...,...
1571,T: hi miriam </s>S: hello clara </s>T: how can...
1572,T: hi again sabrina how can i help?! </s>S: i'...
1573,T: hi morgan </s>S: hi </s>T: so for this one ...
1574,T: hi kofi i m lina one of the tutors at eedi ...


## 7. Get only the tutor turns

In [74]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import get_tutor_turn_only_EEDI_MathmentorMR

##### MathtutorMR

In [75]:
mathtutormr_df__only_tutor_turns = get_tutor_turn_only_EEDI_MathmentorMR(compressed_mathtutormr_df)
mathtutormr_df__only_tutor_turns

,text
0,simplification?
1,how about combining all of those constants fir...
2,that doesnt sound right thonk i think 12 3 is ...
3,that cubic would not be that hard to factor.
4,that -9 will disappear you didn't differentiat...
...,...
927433,helpers please send help. radhika33 we're goin...
927434,is this statistics?
927435,are the prizes identical?
927436,then every 3-element subset of the 100 people ...


##### EEDI

In [76]:
eedi_df__only_tutor_turns = get_tutor_turn_only_EEDI_MathmentorMR(compressed_eedi_df)
eedi_df__only_tutor_turns

,text
0,"hello lina, just wanted to check, you ok? you ..."
1,"of course, thank you for correcting me on your..."
2,do you have any thoughts about what either ale...
3,"ok. so if alex is wrong, what do you think wou..."
4,what do you think 5.4598 rounds to to 1dp?
...,...
17198,"ok, so there a couple of different ways one wa..."
17199,"ok, so we have 140 what do we get if we divide..."
17200,fab so 10 of 140 is 14 now we need to double t...
17201,we'd have got the same answer if we just did 1...


## 8. Get preceding -1

In [77]:
import final__clean_transform_datasets, importlib
importlib.reload(final__clean_transform_datasets)
from final__clean_transform_datasets import get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR

##### mathtutormr

In [78]:
# with speaker info
mathtutormr__preceding_1_turn__with_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_mathtutormr_df, n=1, include_speaker=True)
mathtutormr__preceding_1_turn__with_speaker_info

,text
0,S: 36? </s>T: that doesnt sound right thonk i ...
1,S: yeah </s>T: integral of 1 x from -1 to 1 i ...
2,"S: alright, theyre suuper messy though and in ..."
3,S: uhh i think u got it wrong but ok ill solve...
4,S: ohhhhh thanks sneaky </s>T: no worries mate
...,...
115590,S: not really how do i write expression x </s>...
115591,S: but how do i use the given? </s>T: attempt ...
115592,S: did i need to round up </s>T: 9000 7 gives ...
115593,S: thx! have a nice day </s>T: i use it all th...


In [79]:
# without speaker info
mathtutormr__preceding_1_turn__without_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_mathtutormr_df, n=1, include_speaker=False)
mathtutormr__preceding_1_turn__without_speaker_info

,text
0,36? </s>that doesnt sound right thonk i think ...
1,yeah </s>integral of 1 x from -1 to 1 i tried ...
2,"alright, theyre suuper messy though and in swe..."
3,uhh i think u got it wrong but ok ill solve it...
4,ohhhhh thanks sneaky </s>no worries mate
...,...
115590,not really how do i write expression x </s>wha...
115591,but how do i use the given? </s>attempt to use it
115592,did i need to round up </s>9000 7 gives you ho...
115593,thx! have a nice day </s>i use it all the time...


##### eedi

In [80]:
# with speaker info
eedi_df__preceding_1_turn__with_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_eedi_df, n=1, include_speaker=True)
eedi_df__preceding_1_turn__with_speaker_info

,text
0,S: ok thank you so much take care enjoy the re...
1,S: yup </s>T: no problem!
2,"S: is it b then </s>T: yes, b bye!"
3,S: yes thankyou </s>T: great! well done! bye f...
4,S: yes. </s>T: of course! nice one for request...
...,...
1571,"S: yes thank you, see you later </s>T: nice on..."
1572,S: yep </s>T: bye for now
1573,"S: okay i know </s>T: alright, i'll pass you b..."
1574,S: okay </s>T: i ll pass you back to eedi to c...


In [81]:
# without speaker info
eedi__preceding_1_turn__without_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_eedi_df, n=1, include_speaker=False)
eedi__preceding_1_turn__without_speaker_info

,text
0,ok thank you so much take care enjoy the rest ...
1,yup </s>no problem!
2,"is it b then </s>yes, b bye!"
3,yes thankyou </s>great! well done! bye for now
4,yes. </s>of course! nice one for requesting he...
...,...
1571,"yes thank you, see you later </s>nice one for ..."
1572,yep </s>bye for now
1573,"okay i know </s>alright, i'll pass you back to..."
1574,okay </s>i ll pass you back to eedi to continu...


In [82]:
# without speaker info
mathtutormr__preceding_1_turn__without_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_mathtutormr_df, n=2, include_speaker=False)
mathtutormr__preceding_1_turn__without_speaker_info

,text
0,how about combining all of those constants fir...
1,that -9 will disappear you didn't differentiat...
2,"i'm actually at a meeting at work lmao, send m..."
3,split the middle term x 2 x x 1. ... now you c...
4,"choose the last number first, and one of the o..."
...,...
115590,what about this one </s>not really how do i wr...
115591,"why the weird angle? however, like most logica..."
115592,oh there's where you went wrong </s>did i need...
115593,it's free you can paste in whatever you want a...


## 9. Get preceding -2 turns

##### mathtutormr

In [83]:
# with speaker info
mathtutormr__preceding_2_turn__with_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_mathtutormr_df, n=2, include_speaker=True)
mathtutormr__preceding_2_turn__with_speaker_info

,text
0,T: how about combining all of those constants ...
1,T: that -9 will disappear you didn't different...
2,"T: i'm actually at a meeting at work lmao, sen..."
3,T: split the middle term x 2 x x 1. ... now yo...
4,"T: choose the last number first, and one of th..."
...,...
115590,T: what about this one </s>S: not really how d...
115591,"T: why the weird angle? however, like most log..."
115592,T: oh there's where you went wrong </s>S: did ...
115593,T: it's free you can paste in whatever you wan...


In [84]:
# without speaker info
mathtutormr__preceding_2_turn__without_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_mathtutormr_df, n=2, include_speaker=False)
mathtutormr__preceding_2_turn__without_speaker_info

,text
0,how about combining all of those constants fir...
1,that -9 will disappear you didn't differentiat...
2,"i'm actually at a meeting at work lmao, send m..."
3,split the middle term x 2 x x 1. ... now you c...
4,"choose the last number first, and one of the o..."
...,...
115590,what about this one </s>not really how do i wr...
115591,"why the weird angle? however, like most logica..."
115592,oh there's where you went wrong </s>did i need...
115593,it's free you can paste in whatever you want a...


eedi

In [85]:
# with speaker info
eedi_df__preceding_2_turn__with_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_eedi_df, n=2, include_speaker=True)
eedi_df__preceding_2_turn__with_speaker_info

,text
0,"T: to 1 decimal place, it's 5.5 </s>S: ok than..."
1,T: are you ready to go back and submit your an...
2,T: yep. i just wanted you to see why! </s>S: i...
3,T: great! does this all make sense? </s>S: yes...
4,T: so pleased to hear ready to go back to eedi...
...,...
1571,T: you are super welcome ready to go back to e...
1572,T: ready to go back to eedi? </s>S: yep </s>T:...
1573,"T: url it's not homework, they are just quizze..."
1574,T: you are super welcome this quiz might be he...


In [86]:
# without speaker info
eedi_df__preceding_2_turn__without_speaker_info = get_all_turn__preceding_n_turn__utterances_EEDI_MathmentorMR(filtered_compressed_eedi_df, n=2, include_speaker=True)
eedi_df__preceding_2_turn__without_speaker_info

,text
0,"T: to 1 decimal place, it's 5.5 </s>S: ok than..."
1,T: are you ready to go back and submit your an...
2,T: yep. i just wanted you to see why! </s>S: i...
3,T: great! does this all make sense? </s>S: yes...
4,T: so pleased to hear ready to go back to eedi...
...,...
1571,T: you are super welcome ready to go back to e...
1572,T: ready to go back to eedi? </s>S: yep </s>T:...
1573,"T: url it's not homework, they are just quizze..."
1574,T: you are super welcome this quiz might be he...


## 9. Concatenate and Save Datasets
The pretraining will be done using the turns with speaker info

In [87]:
# # all turns
# mathtutormr_df__all_turns__without_speaker_info
# eedi_df__all_turns__without_speaker_info

# mathtutormr_df__all_turns__with_speaker_info
# eedi_df__all_turns__with_speaker_info

# # tutor turn only
# mathtutormr_df__only_tutor_turns
# eedi_df__only_tutor_turns

# # preceding -1 turn
# mathtutormr__preceding_1_turn__with_speaker_info
# eedi_df__preceding_1_turn__with_speaker_info

# mathtutormr__preceding_1_turn__without_speaker_info
# eedi_df__preceding_1_turn__without_speaker_info

# # preceding -2 turn
# mathtutormr__preceding_2_turn__with_speaker_info
# eedi_df__preceding_2_turn__with_speaker_info

# mathtutormr__preceding_2_turn__without_speaker_info
# eedi_df__preceding_2_turn__without_speaker_info

In [89]:
import os

# Create the directory if it doesn't exist
os.makedirs("./data/pretraining_data", exist_ok=True)


# Concatenate mathdial and eedi dataset
mathtutormr_eedi__all_turns__with_speaker_info = pd.concat([mathtutormr_df__all_turns__with_speaker_info, eedi_df__all_turns__with_speaker_info])
mathtutormr_eedi__all_turns__without_speaker_info = pd.concat([mathtutormr_df__all_turns__without_speaker_info, eedi_df__all_turns__without_speaker_info])

mathtutormr_eedi__tutor_turn_only = pd.concat([mathtutormr_df__only_tutor_turns, eedi_df__only_tutor_turns])

mathtutormr_eedi__preceding_1_turn__with_speaker_info = pd.concat([mathtutormr__preceding_1_turn__with_speaker_info, eedi_df__preceding_1_turn__with_speaker_info])
mathtutormr_eedi__preceding_1_turn__without_speaker_info = pd.concat([mathtutormr__preceding_1_turn__without_speaker_info, eedi__preceding_1_turn__without_speaker_info])

mathtutormr_eedi__preceding_2_turn__with_speaker_info = pd.concat([mathtutormr__preceding_2_turn__with_speaker_info, eedi_df__preceding_2_turn__with_speaker_info])
mathtutormr_eedi__preceding_2_turn__without_speaker_info = pd.concat([mathtutormr__preceding_2_turn__without_speaker_info, eedi_df__preceding_2_turn__without_speaker_info])


# Save data
mathtutormr_eedi__all_turns__with_speaker_info.to_csv("./data/pretraining_data/mathtutormr_eedi__all_turns__with_speaker_info.csv", index=False)
mathtutormr_eedi__all_turns__without_speaker_info.to_csv("./data/pretraining_data/mathtutormr_eedi__all_turns__without_speaker_info.csv", index=False)

mathtutormr_eedi__preceding_1_turn__with_speaker_info.to_csv("./data/pretraining_data/mathtutormr_eedi__preceding_1_turn__with_speaker_info.csv", index=False)
mathtutormr_eedi__preceding_1_turn__without_speaker_info.to_csv("./data/pretraining_data/mathtutormr_eedi__preceding_1_turn__without_speaker_info.csv", index=False)

mathtutormr_eedi__preceding_2_turn__with_speaker_info.to_csv("./data/pretraining_data/mathtutormr_eedi__preceding_2_turn__with_speaker_info.csv", index=False)
mathtutormr_eedi__preceding_2_turn__without_speaker_info.to_csv("./data/pretraining_data/mathtutormr_eedi__preceding_2_turn__without_speaker_info.csv", index=False)

mathtutormr_eedi__tutor_turn_only.to_csv("./data/pretraining_data/mathtutormr_eedi__tutor_turn_only.csv", index=False)

In [1]:
import pandas as pd

In [5]:
mathtutormr_eedi__all_turns__with_speaker_info = pd.read_csv("./data/pretraining_data/mathtutormr_eedi__all_turns__with_speaker_info.csv")
len(mathtutormr_eedi__all_turns__with_speaker_info)

117171

In [6]:
mathtutormr_eedi__preceding_1_turn__without_speaker_info = pd.read_csv("./data/pretraining_data/mathtutormr_eedi__preceding_1_turn__without_speaker_info.csv")
len(mathtutormr_eedi__preceding_1_turn__without_speaker_info)

117171

In [7]:
mathtutormr_eedi__tutor_turn_only = pd.read_csv("./data/pretraining_data/mathtutormr_eedi__tutor_turn_only.csv")
len(mathtutormr_eedi__tutor_turn_only)

944641